In [1]:
!pip install boto3 pandas pyarrow fsspec s3fs

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/

In [ ]:
# import io
# import json
import boto3
import pandas as pd
# from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://seaweedfs:8333",
    aws_access_key_id="admin",
    aws_secret_access_key="secret"
)

# s3.create_bucket(Bucket="bronze")
# s3.create_bucket(Bucket="silver")
# s3.create_bucket(Bucket="gold")

In [ ]:
df = pd.read_csv('sales2.csv', sep=';')
df

In [ ]:
df.dtypes

In [ ]:
import io
from botocore.client import Config

# Schema yang diharapkan
schema = {
    "id": "int64",
    "store": "str",
    "product": "str",
    "qty": "int64",
    "price": "int64"
}

actual_schema = df.dtypes.astype(str).to_dict()
try:
    if actual_schema != schema:
        print("Schema tidak sesuai!\n")
        raise ValueError("Schema Berbeda")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False)
    s3 = boto3.client(
        "s3",
        endpoint_url="http://seaweedfs:8333",
        aws_access_key_id="admin",
        aws_secret_access_key="secret",
        config=Config(
            s3={"addressing_style": "path"},
            request_checksum_calculation="when_required"
        )
    )
    s3.put_object(
        Bucket="bronze",
        Key="sales.parquet",
        Body=buffer.getvalue()
    )
    print("Sesuai schema!")
    print("Data behasil masuk Bronze!")
except ValueError as e:
    print("\nError:", e)

In [38]:
#1 Skenariokan data actual berbeda dengan schema

#2 Skenariokan data actual dan schema berupa tipe data dan wajib diisi

# Tambahkan data di sales2.csv yang kolom pricenya null, sementara di data contract dibuat aturan kolom price wajib diisi dan sesuai tipe data